### Local: Hunyuan3D-2

In [6]:
# https://github.com/Tencent-Hunyuan/Hunyuan3D-2

In [7]:
import os
os.environ["HF_HOME"] = "D:/huggingface"

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
import time

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("D:/huggingface/Hunyuan3D-2")
print("Модель загружена!")

KeyboardInterrupt: 

In [ ]:
import time

t0 = time.time()
mesh = pipeline(image="OUTPUT/img03_clean.png")[0]
elapsed = time.time() - t0

mesh.export("OUTPUT/img03_mesh.glb")
print(f"Меш сохранён: OUTPUT/img03_mesh.glb")
print(f"Время: {elapsed:.1f}с")
print(f"Вершин: {len(mesh.vertices)}, граней: {len(mesh.faces)}")


Volume Decoding: 100%|██████████| 7134/7134 [00:33<00:00, 216.12it/s]


Меш сохранён: OUTPUT/img03_mesh.glb
Время: 52.0с
Вершин: 265974, граней: 531904


In [ ]:
import trimesh
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import art3d

mesh = trimesh.load("OUTPUT/img03_mesh.glb", force="mesh")
vertices = np.array(mesh.vertices)
faces = np.array(mesh.faces)

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')

poly = art3d.Poly3DCollection(vertices[faces], alpha=0.9, edgecolor='gray', linewidth=0.1)
poly.set_facecolor('lightblue')
ax.add_collection3d(poly)

scale = (vertices.max(axis=0) - vertices.min(axis=0)).max() / 2
mid = vertices.mean(axis=0)
ax.set_xlim(mid[0] - scale, mid[0] + scale)
ax.set_ylim(mid[1] - scale, mid[1] + scale)
ax.set_zlim(mid[2] - scale, mid[2] + scale)
ax.view_init(elev=90, azim=270)
ax.axis("off")

plt.savefig("OUTPUT/img03_mesh_preview.png", dpi=150, bbox_inches="tight", pad_inches=0)
print("Скриншот: OUTPUT/img03_mesh_preview.png")
plt.show()

Скриншот: OUTPUT/img03_mesh_preview.png


C:\Users\Luga\AppData\Local\Temp\ipykernel_3828\1250589130.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### fal.ai: Trellis
- ~$0.02

In [1]:
import os, time
import fal_client
import requests
from dotenv import load_dotenv

load_dotenv()
os.environ["FAL_KEY"] = os.getenv("FAL_API_KEY", "")

image_url = fal_client.upload_file("OUTPUT/img03_clean.png")

t0 = time.time()
result = fal_client.subscribe("fal-ai/trellis", arguments={
    "image_url": image_url,
})
elapsed = time.time() - t0

glb_url = result["model_mesh"]["url"]
glb_bytes = requests.get(glb_url).content

with open("OUTPUT/img03_mesh_trellis.glb", "wb") as f:
    f.write(glb_bytes)

print(f"Время: {elapsed:.1f}с")
print(f"Меш: {len(glb_bytes) / 1024 / 1024:.1f} MB")

Время: 21.6с
Меш: 1.3 MB


In [5]:
import numpy as np
import imageio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Конвертируем текстуру в vertex colors
if hasattr(mesh.visual, "to_color"):
    mesh.visual = mesh.visual.to_color()

face_colors = mesh.visual.vertex_colors[mesh.faces].mean(axis=1) / 255.0

verts = mesh.vertices - mesh.vertices.mean(axis=0)
verts /= np.abs(verts).max()
verts = verts[:, [0, 2, 1]]

gif_frames = []
for angle in range(0, 360, 10):
    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection="3d")
    ax.view_init(elev=20, azim=angle)
    polys = verts[mesh.faces]
    collection = Poly3DCollection(polys, linewidth=0, edgecolor="none")
    collection.set_facecolor(face_colors[:, :3])
    ax.add_collection3d(collection)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.axis("off")
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    w, h = fig.canvas.get_width_height()
    gif_frames.append(buf.reshape(h, w, 4)[:, :, :3].copy())
    plt.close(fig)

imageio.mimsave("OUTPUT/img03_mesh_trellis.gif", gif_frames, fps=8, loop=0)
print(f"GIF: {os.path.getsize('OUTPUT/img03_mesh_trellis.gif') / 1024 / 1024:.1f} MB")


GIF: 0.5 MB
